In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv('kc_house_data.csv')

# Rename 'id' to 'house_id'
df.rename(columns={'id': 'house_id'}, inplace=True)

# Calculate 'age' from 'yr_built'
df['age'] = 2025 - df['yr_built']

# Map 'condition' to 'condition_name'
condition_mapping = {
    1: 'Poor',
    2: 'Fair',
    3: 'Average',
    4: 'Good',
    5: 'Very Good'
}
df['condition_name'] = df['condition'].map(condition_mapping)

houses = df[['house_id', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot',
          'floors', 'waterfront', 'age', 'view', 'condition_name', 'price']]


In [3]:
df = houses.copy()
df


,house_id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,age,view,condition_name,price
0,7129300520,3,1.00,1180,5650,1.0,0,70,0,Average,221900.0
1,6414100192,3,2.25,2570,7242,2.0,0,74,0,Average,538000.0
2,5631500400,2,1.00,770,10000,1.0,0,92,0,Average,180000.0
3,2487200875,4,3.00,1960,5000,1.0,0,60,0,Very Good,604000.0
4,1954400510,3,2.00,1680,8080,1.0,0,38,0,Average,510000.0
...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,3,2.50,1530,1131,3.0,0,16,0,Average,360000.0
21609,6600060120,4,2.50,2310,5813,2.0,0,11,0,Average,400000.0
21610,1523300141,2,0.75,1020,1350,2.0,0,16,0,Average,402101.0
21611,291310100,3,2.50,1600,2388,2.0,0,21,0,Average,400000.0


## Data Preprocessing

### Imputing

In [4]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='mean')
df['age'] = imputer.fit_transform(df[['age']]).astype(int)
df


,house_id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,age,view,condition_name,price
0,7129300520,3,1.00,1180,5650,1.0,0,70,0,Average,221900.0
1,6414100192,3,2.25,2570,7242,2.0,0,74,0,Average,538000.0
2,5631500400,2,1.00,770,10000,1.0,0,92,0,Average,180000.0
3,2487200875,4,3.00,1960,5000,1.0,0,60,0,Very Good,604000.0
4,1954400510,3,2.00,1680,8080,1.0,0,38,0,Average,510000.0
...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,3,2.50,1530,1131,3.0,0,16,0,Average,360000.0
21609,6600060120,4,2.50,2310,5813,2.0,0,11,0,Average,400000.0
21610,1523300141,2,0.75,1020,1350,2.0,0,16,0,Average,402101.0
21611,291310100,3,2.50,1600,2388,2.0,0,21,0,Average,400000.0


### Normalization

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[['sqft_living', 'sqft_lot']] = scaler.fit_transform(df[['sqft_living', 'sqft_lot']])
df


,house_id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,age,view,condition_name,price
0,7129300520,3,1.00,-0.979835,-0.228321,1.0,0,70,0,Average,221900.0
1,6414100192,3,2.25,0.533634,-0.189885,2.0,0,74,0,Average,538000.0
2,5631500400,2,1.00,-1.426254,-0.123298,1.0,0,92,0,Average,180000.0
3,2487200875,4,3.00,-0.130550,-0.244014,1.0,0,60,0,Very Good,604000.0
4,1954400510,3,2.00,-0.435422,-0.169653,1.0,0,38,0,Average,510000.0
...,...,...,...,...,...,...,...,...,...,...,...
21608,263000018,3,2.50,-0.598746,-0.337424,3.0,0,16,0,Average,360000.0
21609,6600060120,4,2.50,0.250539,-0.224386,2.0,0,11,0,Average,400000.0
21610,1523300141,2,0.75,-1.154047,-0.332137,2.0,0,16,0,Average,402101.0
21611,291310100,3,2.50,-0.522528,-0.307076,2.0,0,21,0,Average,400000.0


### Encoding

In [6]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()
df['condition'] = encoder.fit_transform(df[['condition_name']])
df.drop(['condition_name'], axis=1, inplace=True)


## Model Development

In [7]:
df.head()


,house_id,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,age,view,price,condition
0,7129300520,3,1.00,-0.979835,-0.228321,1.0,0,70,0,221900.0,0.0
1,6414100192,3,2.25,0.533634,-0.189885,2.0,0,74,0,538000.0,0.0
2,5631500400,2,1.00,-1.426254,-0.123298,1.0,0,92,0,180000.0,0.0
3,2487200875,4,3.00,-0.130550,-0.244014,1.0,0,60,0,604000.0,4.0
4,1954400510,3,2.00,-0.435422,-0.169653,1.0,0,38,0,510000.0,0.0


In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

# Models to test
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor


# === Load and prepare data ===
train_val_df, test_df = train_test_split(df, test_size=0.15, random_state=43)
train_df, val_df = train_test_split(train_val_df, test_size=0.15, random_state=43)
# train_df = pd.read_csv('train.csv')
# val_df = pd.read_csv('val.csv')
# test_df = pd.read_csv('test.csv')

TARGET = 'price'

X_train = train_df.drop(columns=[TARGET, 'house_id'])
y_train = train_df[TARGET]
X_val = val_df.drop(columns=[TARGET, 'house_id'])
y_val = val_df[TARGET]
X_test = test_df.drop(columns=[TARGET, 'house_id'])


In [9]:
X_test


,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,age,view,condition
9053,4,2.75,1.426472,0.025255,1.0,0,64,0,2.0
11461,3,1.50,-0.990723,-0.336386,2.0,0,18,0,0.0
3508,4,2.50,1.012718,-0.089667,2.0,0,20,0,0.0
21600,5,3.75,2.602405,-0.169460,2.0,0,17,0,0.0
9728,3,1.00,-0.870952,-0.128731,1.0,0,61,0,0.0
...,...,...,...,...,...,...,...,...,...
13684,4,2.50,0.882059,-0.000965,2.0,0,22,0,0.0
13341,3,2.25,-0.293874,-0.180518,1.0,0,66,0,2.0
12680,3,1.00,-0.511639,1.044507,1.0,0,47,0,0.0
12219,4,2.00,0.043662,-0.154660,1.5,0,65,0,2.0


## Model Development

### Regression

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error

# === 1. Param grids (list-based only) ===
param_grids = {
    'Random Forest': {
        'model': RandomForestRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [None, 10, 20, 30],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingRegressor(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'subsample': [0.8, 0.9, 1.0]
        }
    },
    'Ridge': {
        'model': Ridge(random_state=42),
        'params': {
            'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
            'solver': ['auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg']
        }
    },
    'Lasso': {
        'model': Lasso(random_state=42),
        'params': {
            'alpha': [0.001, 0.01, 0.1, 1.0, 10.0],
            'max_iter': [1000, 2000, 5000]
        }
    },
    'ElasticNet': {
        'model': ElasticNet(random_state=42),
        'params': {
            'alpha': [0.001, 0.01, 0.1, 1.0],
            'l1_ratio': [0.1, 0.5, 0.7, 0.9],
            'max_iter': [1000, 2000]
        }
    }
}

# === 2. Train & evaluate using MAE ===
best_model = None
lowest_mae = float('inf')

for name, config in param_grids.items():
    model = config['model']
    params = config['params']

    print(f"\n🔍 Training: {name} with RandomizedSearchCV (scoring = MAE)")
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=20,  # Number of combinations to sample
        scoring='neg_mean_absolute_error',
        cv=3,
        random_state=99,
        n_jobs=-1
    )
    random_search.fit(X_train, y_train)
    best = random_search.best_estimator_

    preds = best.predict(X_val)
    mae = mean_absolute_error(y_val, preds)
    print(f"✅ {name} MAE: {mae:.2f}")

    if mae < lowest_mae:
        lowest_mae = mae
        best_model = best
        final_test_input = X_test

# === 3. Predict on test and save ===
final_preds = best_model.predict(final_test_input)
output = pd.DataFrame({
    "house_id": test_df["house_id"],
    "price": final_preds
})
output.to_csv("predictions.csv", index=False)
print("\n📦 predictions.csv saved with best model's predictions.")



🔍 Training: Random Forest with RandomizedSearchCV (scoring = MAE)
✅ Random Forest MAE: 135453.90

🔍 Training: Gradient Boosting with RandomizedSearchCV (scoring = MAE)
✅ Gradient Boosting MAE: 133613.05

🔍 Training: Ridge with RandomizedSearchCV (scoring = MAE)
✅ Ridge MAE: 150962.35

🔍 Training: Lasso with RandomizedSearchCV (scoring = MAE)


/opt/anaconda3/envs/endgame/lib/python3.12/site-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 15 is smaller than n_iter=20. Running 15 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


✅ Lasso MAE: 150963.60

🔍 Training: ElasticNet with RandomizedSearchCV (scoring = MAE)
✅ ElasticNet MAE: 149565.80

📦 predictions.csv saved with best model's predictions.


### Classification

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import f1_score

# === 1. Sample classification dataset ===
X, y = make_classification(
    n_samples=1000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_classes=2,
    weights=[0.7, 0.3],  # simulate imbalance
    random_state=42
)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# === 2. Param grids (no scipy, just lists) ===
param_grids = {
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200, 300],
            'max_depth': [None, 10, 20],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2]
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'learning_rate': [0.01, 0.1],
            'max_depth': [3, 5]
        }
    },
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'params': {
            'C': [0.01, 0.1, 1.0, 10.0],
            'solver': ['liblinear', 'lbfgs']
        }
    },
    'SVM': {
        'model': SVC(probability=True, random_state=42),
        'params': {
            'C': [0.1, 1, 10],
            'kernel': ['linear', 'rbf'],
            'gamma': ['scale', 'auto']
        }
    }
}

# === 3. Train & evaluate using F1 ===
best_model = None
best_f1 = 0

for name, config in param_grids.items():
    model = config['model']
    params = config['params']

    print(f"\n🔍 Training: {name} (scoring = F1)")
    random_search = RandomizedSearchCV(
        estimator=model,
        param_distributions=params,
        n_iter=10,
        scoring='f1',
        cv=3,
        random_state=42,
        n_jobs=-1
    )
    random_search.fit(X_train, y_train)
    best = random_search.best_estimator_

    preds = best.predict(X_val)
    f1 = f1_score(y_val, preds)
    print(f"✅ {name} F1 Score: {f1:.3f}")

    if f1 > best_f1:
        best_f1 = f1
        best_model = best
        final_test_input = X_test

# === 4. Final prediction on test set ===
final_preds = best_model.predict(final_test_input)
final_f1 = f1_score(y_test, final_preds)
print(f"\n🏁 Final Test F1 Score: {final_f1:.3f}")
